This notebook estimates heat exposure for Citi Bike users in New York City by combining trip data, precomputed bicycle routes, and high-resolution thermal data. It begins by cleaning and preparing Citi Bike trip records and loading Valhalla-generated routes between station pairs. Each trip is matched to its corresponding route, and a cleaned street network is used as the spatial reference for analysis. Hourly UTCI raster data are then applied to street segments to assign heat exposure values across time. For each hour, the notebook filters trips, attaches route geometries, and identifies which street segments experience high heat stress. It counts how frequently these high-UTCI segments are used by cyclists and computes trip-level and system-wide exposure metrics. Finally, it aggregates these results to produce geospatial outputs highlighting the most frequently used high-heat street segments, enabling identification of urban heat exposure hotspots for micromobility users.

## 1. Setup and File Paths

This section imports the required Python libraries and defines the main input and output directories used throughout the hotspot analysis. The workflow combines Citi Bike trip data, precomputed Valhalla bicycle routes, NEAT street geometries, and hourly UTCI raster outputs.

In [1]:
# ─── 1 · SETUP ──────────────────────────────────────────────────────
import os
import glob
import zipfile
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
from rasterstats import zonal_stats
import matplotlib.pyplot as plt

# Paths
WORK        = Path(r"C:\valhalla_neat")
TRIP_DIR    = Path(
    r"C:\Users\Agustin\Documents\SummerLab\SOILWEG\INPUT_DATA"
    r"\SEMESTER\SPRING2025\biketrip-heat-exposure\trip_data"
)
UTCI_DIR    = Path(
    r"C:\Users\Agustin\Documents\SummerLab\SOILWEG\INPUT_DATA"
    r"\SEMESTER\TREE_ANALYSIS\nyc_utci_maps\tree_base"
)
RESULTS_DIR = Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("→ outputs →", RESULTS_DIR)


→ outputs → C:\Users\Agustin\Documents\2025summer\sanity_check\results


## 2. Load and Clean Citi Bike Trip Data

This section loads Citi Bike trip records from monthly ZIP archives, keeps the fields needed for station-pair routing and temporal filtering, removes incomplete records, converts trip start times to datetime format, and creates a station coordinate lookup table.

In [2]:
# ─── 2 · LOAD TRIPS & COLLAPSE COORDS  (reads *every* CSV in every ZIP) ─────────
import zipfile
import pandas as pd
from pathlib import Path

# Path to your trip‐data folder (June 2024 ZIP is in there).
TRIP_DIR = Path(
    r"C:\Users\Agustin\Documents\SummerLab\SOILWEG\INPUT_DATA"
    r"\SEMESTER\SPRING2025\biketrip-heat-exposure\trip_data"
)

# Include 'ride_id' so we carry the true trip identifier downstream
USE = [
    "ride_id",
    "started_at","start_station_id","end_station_id",
    "start_lat","start_lng","end_lat","end_lng"
]

# Force floats to float32; strings to “string”
DTYPE = {c: "float32" for c in USE if c.endswith(("_lat","_lng"))}
DTYPE.update({
    "ride_id":            "string",
    "started_at":         "string",
    "start_station_id":   "string",
    "end_station_id":     "string"
})

def read_trips(zp: Path) -> pd.DataFrame:
    with zipfile.ZipFile(zp) as z:
        csv_parts = [f for f in z.namelist() if f.endswith(".csv")]
        frames = []
        for csv_fname in csv_parts:
            df_part = pd.read_csv(
                z.open(csv_fname),
                usecols=USE,
                dtype=DTYPE,
                low_memory=False
            )
            frames.append(df_part)
    return pd.concat(frames, ignore_index=True)

# Gather and list all monthly ZIPs
zip_paths = sorted(TRIP_DIR.glob("*.zip"))
print(f"► found {len(zip_paths)} ZIP archive(s) in trip_data:")
for z in zip_paths:
    print("   •", z.name)

# Read each ZIP and concatenate
frames = []
for z in zip_paths:
    df_z = read_trips(z)
    print(f"{z.name:35s} → {len(df_z):,} rows before dropna")
    frames.append(df_z)

trips = pd.concat(frames, ignore_index=True)
print(f"\n► TOTAL TRIPS LOADED: {len(trips):,} rows before dropna\n")

# Clean & drop
trips.replace(r"^\s*$", pd.NA, regex=True, inplace=True)
pre = len(trips)
trips.dropna(inplace=True)
post = len(trips)
print(f"► AFTER dropna: {post:,} rows remain  (dropped {pre-post:,})\n")

# Convert started_at to datetime
trips["started_at"] = pd.to_datetime(trips["started_at"])

# Compute canonical station coordinates
stations = (
    pd.concat([
        trips[["start_station_id","start_lat","start_lng"]]
             .rename(columns={
                 "start_station_id":"sid",
                 "start_lat":"lat","start_lng":"lon"
             }),
        trips[["end_station_id","end_lat","end_lng"]]
             .rename(columns={
                 "end_station_id":"sid",
                 "end_lat":"lat","end_lng":"lon"
             })
    ])
    .groupby("sid").median().reset_index()
)

lat_map = stations.set_index("sid")["lat"]
lon_map = stations.set_index("sid")["lon"]

for col, mapper in [
    ("start_lat", lat_map),
    ("start_lng", lon_map),
    ("end_lat",   lat_map),
    ("end_lng",   lon_map)
]:
    trips[col] = trips[col
                       .replace("_lat","_station_id")
                       .replace("_lng","_station_id")
                      ].map(mapper)

print(f"► {len(stations):,} unique stations after coordinate collapse")


► found 1 ZIP archive(s) in trip_data:
   • 202406-citibike-tripdata.zip
202406-citibike-tripdata.zip        → 4,783,576 rows before dropna

► TOTAL TRIPS LOADED: 4,783,576 rows before dropna

► AFTER dropna: 4,768,748 rows remain  (dropped 14,828)

► 2,309 unique stations after coordinate collapse


## 3. Load Precomputed Valhalla Routes

This section loads the precomputed bicycle routes between Citi Bike station pairs. These route geometries are later attached to individual trips using the original start and end station IDs.

In [3]:
# ─── 3 · LOAD PRE-COMPUTED ROUTES ──────────────────────────────────
route_gpkg = WORK / "verified_station_pair_routes_fixed.gpkg"
routes = (
    gpd.read_file(route_gpkg, layer="routes_with_correct_ids")
       .to_crs(4326)[["orig_start","orig_end","geometry"]]
)
print("routes table loaded:", len(routes))


routes table loaded: 738979


## 4. Optional Route Coverage Checks

This section checks whether the precomputed route file is compatible with the trip table. It verifies route data types, counts unique station pairs in the raw trips and route file, and estimates how many trips are dropped when routes are attached.

4.1 Station-Pair Route Coverage

In [6]:
# Cell 4.1 · How many unique station‐pairs in raw trips? 
#           How many in routes? How many overlap?

# 1) Unique pairs in trips (after dropna in Cell 2)
pair_trips = (
    trips[["start_station_id","end_station_id"]]
    .astype("string")
    .drop_duplicates()
)
n_trip_pairs = len(pair_trips)
print(f"► Unique station-pairs in RAW trips: {n_trip_pairs:,}")

# 2) Unique pairs in routes (from the full E-FULL output)
pair_routes = (
    routes[["orig_start","orig_end"]]
    .astype("string")
    .drop_duplicates()
)
n_route_pairs = len(pair_routes)
print(f"► Unique station-pairs in ROUTES table: {n_route_pairs:,}")

# 3) How many of those trip-pairs actually appear in the routes table?
merged_pairs = pair_trips.merge(
    pair_routes,
    left_on=["start_station_id","end_station_id"],
    right_on=["orig_start","orig_end"],
    how="inner"
)
n_matched_pairs = len(merged_pairs)
print(f"► Matching station-pairs (trips ∩ routes): {n_matched_pairs:,}")

# 4) How many trip-pairs did NOT find a route?
n_missed_pairs = n_trip_pairs - n_matched_pairs
print(f"► Station-pairs in trips but NOT in routes: {n_missed_pairs:,}\n")

# 5) Example missing station-pairs (first 10)
print("▶ Example missing station-pairs:")
missing = pair_trips.merge(
    pair_routes,
    left_on=["start_station_id","end_station_id"],
    right_on=["orig_start","orig_end"],
    how="left",
    indicator=True
).query('_merge == "left_only"')[["start_station_id","end_station_id"]]

print(missing.head(10).to_string(index=False))


► Unique station-pairs in RAW trips: 736,630
► Unique station-pairs in ROUTES table: 738,979
► Matching station-pairs (trips ∩ routes): 736,422
► Station-pairs in trips but NOT in routes: 208

▶ Example missing station-pairs:
start_station_id end_station_id
         4680.05   Shop Morgan 
         4856.05   Shop Morgan 
         4161.01   Shop Morgan 
         5294.04   Shop Morgan 
         6700.14          HB402
         5068.02   Shop Morgan 
         5847.01   Shop Morgan 
         4212.01   Shop Morgan 
         4066.15   Shop Morgan 
         4640.01   Shop Morgan 


4.2 Trip-Level Route Match Rate

In [7]:
# Cell 4.2 · Exactly how many trips drop out when you do attach_routes

# 1) Tag every trip whether it finds a route
trips_with_flag = trips.merge(
    routes[["orig_start","orig_end"]],        # use routes loaded in Section 3
    left_on=["start_station_id","end_station_id"],
    right_on=["orig_start","orig_end"],
    how="left",
    indicator=True
)

n_total_trips   = len(trips_with_flag)
n_matched_trips = (trips_with_flag["_merge"] == "both").sum()
n_dropped_trips = (trips_with_flag["_merge"] == "left_only").sum()

print(f"► Total raw trips:             {n_total_trips:,}")
print(f"► Trips with a matching route: {n_matched_trips:,}")
print(f"► Trips **without** a route:   {n_dropped_trips:,}")

# 2) Example of a few dropped-trip rows
print("\n▶ Example dropped trips (first 5):")
print(
    trips_with_flag[trips_with_flag["_merge"] == "left_only"]
    [["start_station_id","end_station_id","started_at"]]
    .head(5)
    .to_string(index=False)
)


► Total raw trips:             4,768,748
► Trips with a matching route: 4,768,395
► Trips **without** a route:   353

▶ Example dropped trips (first 5):
start_station_id end_station_id              started_at
         4680.05   Shop Morgan  2024-06-17 14:40:00.065
         4856.05   Shop Morgan  2024-06-17 22:58:39.920
         4161.01   Shop Morgan  2024-06-23 06:33:29.671
         5294.04   Shop Morgan  2024-06-17 06:36:56.778
         5294.04   Shop Morgan  2024-06-22 06:40:49.693


## 5. Load Base NEAT Street Network

This section loads the cleaned NEAT street geometry used for UTCI assignment and hotspot counting. Each street segment receives a `seg_id`, which is used as the common identifier across hourly UTCI layers and hotspot outputs.

In [5]:
# ─── 5 · LOAD NEAT STREET GEOMETRY ─────────────────────────────────
streets_base = gpd.read_file(
    r"C:\Users\Agustin\Documents\2025summer\neatnet_generator\verified_streets_neat_bidirectional.shp"
).to_crs(32118)

streets_base["seg_id"] = streets_base.index
street_len = streets_base.length
print("NEAT street geometry:", len(streets_base))


NEAT street geometry: 58472


## 6. Define Helper Functions

This section defines helper functions for locating hourly UTCI rasters, attaching precomputed route geometries to Citi Bike trips, and assigning mean UTCI values from raster cells to NEAT street segments.

In [6]:
# ─── 6 · HELPERS (with neutral index name) ─────────────────────────
BUF, MIN_LEN, MIN_RATIO, HOT = 3, 10, 0.20, 32.0

def utci_path(hour):
    if 0 <= hour <= 20:
        return UTCI_DIR / f"nyc_utci_treebase_173_{hour:02d}00.tif"
    elif hour == 21:
        return UTCI_DIR / "nyc_utci_treebase_172_2100.tif"
    elif hour == 22:
        return UTCI_DIR / "nyc_utci_treebase_172_2200.tif"
    elif hour == 23:
        return UTCI_DIR / "nyc_utci_treebase_172_2300.tif"
    else:
        raise ValueError("hour out of range")

def attach_routes(df):
    """
    Join each trip to its route geometry using the original station IDs
    (orig_start, orig_end). We assume `routes` has columns:
      orig_start, orig_end, geometry
    """
    return df.merge(
        routes[["orig_start","orig_end","geometry"]],
        left_on=["start_station_id","end_station_id"],
        right_on=["orig_start","orig_end"],
        how="left"
    ).dropna(subset=["geometry"])

def street_utci(hour):
    tif = utci_path(hour)
    cache = RESULTS_DIR / f"street_segments_utci_neat_{hour:02d}.gpkg"
    if cache.exists():
        return gpd.read_file(cache).set_index("seg_id")
    s = streets_base.copy()
    print(f"    computing UTCI zonal stats for {tif.name} …")
    zs = zonal_stats(s.geometry, tif, stats=["mean"],
                     all_touched=True, nodata=None)
    s["utci"] = [z["mean"] if z["mean"] is not None else np.nan for z in zs]
    s.to_file(cache, driver="GPKG")
    return s.set_index("seg_id")


## 7. Generate Hourly UTCI Street Segment Layers

This section creates one GeoPackage per hour containing NEAT street segments with mean UTCI values assigned from the corresponding UTCI raster. Existing hourly files are reused to avoid recomputing zonal statistics.

In [14]:
# ─── 7 · GENERATE UTCI‐ANNOTATED NEAT STREETS FOR ALL HOURS ────────────
# Uses your street_utci(hour) helper to produce one GPKG per hour if missing.
for hh in range(24):
    cache = RESULTS_DIR / f"street_segments_utci_neat_{hh:02d}.gpkg"
    if not cache.exists():
        print(f"⏳ Generating {cache.name} …")
        # this will read your simplified shapefile, run zonal_stats on the hh:00 TIFF,
        # attach mean utci values, set seg_id, and write out the GPKG
        street_utci(hh)
    else:
        print(f"✅ Already have {cache.name}")


⏳ Generating street_segments_utci_neat_00.gpkg …
    computing UTCI zonal stats for nyc_utci_treebase_173_0000.tif …
⏳ Generating street_segments_utci_neat_01.gpkg …
    computing UTCI zonal stats for nyc_utci_treebase_173_0100.tif …
⏳ Generating street_segments_utci_neat_02.gpkg …
    computing UTCI zonal stats for nyc_utci_treebase_173_0200.tif …
⏳ Generating street_segments_utci_neat_03.gpkg …
    computing UTCI zonal stats for nyc_utci_treebase_173_0300.tif …
⏳ Generating street_segments_utci_neat_04.gpkg …
    computing UTCI zonal stats for nyc_utci_treebase_173_0400.tif …
⏳ Generating street_segments_utci_neat_05.gpkg …
    computing UTCI zonal stats for nyc_utci_treebase_173_0500.tif …
⏳ Generating street_segments_utci_neat_06.gpkg …
    computing UTCI zonal stats for nyc_utci_treebase_173_0600.tif …
⏳ Generating street_segments_utci_neat_07.gpkg …
    computing UTCI zonal stats for nyc_utci_treebase_173_0700.tif …
⏳ Generating street_segments_utci_neat_08.gpkg …
    computing U

## 8. Process Routes by Hour and Count Hotspot Segments

This section filters Citi Bike trips by start hour, attaches each trip to its precomputed Valhalla route, writes hourly route GeoPackages, counts route overlap with hot UTCI street segments, exports hourly top-100 hotspot layers, and computes trip-level mean UTCI exposure summaries.

In [ ]:
# ─── 8 · PROCESS EVERY HOUR: ROUTES, HOT COUNTS, AND GEODESIC LENGTHS ────────
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Geod
from pathlib import Path

# Geod for WGS84
geod = Geod(ellps="WGS84")

# Constants and directories
BUF       = 3      # buffer for hot‐segment join (m) in projected CRS
MIN_LEN   = 10
MIN_RATIO = 0.20
HOT       = 32.0
RESULTS   = RESULTS_DIR
OUT_DIR   = Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\utci_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Ensure ride_id exists
if "ride_id" not in trips.columns:
    trips["ride_id"] = trips.index.astype(str)

usage_global     = pd.Series(dtype="int", name="count")
usage_global.index.name = "seg_id"
global_mean_recs = []

for hh in range(24):
    hour_str = f"{hh:02d}00"
    print(f"── Hour {hour_str} ──")

    # 1) Attach routes
    trips_hr = trips[trips.started_at.dt.hour == hh].copy()
    if trips_hr.empty:
        print(" no trips"); continue
    trips_hr = attach_routes(trips_hr)
    if trips_hr.empty:
        print(" no matching routes"); continue

    # 2) Build GeoDataFrame in EPSG:4326 for geodesic length
    gdf = gpd.GeoDataFrame(
        trips_hr[["ride_id","geometry"]],
        crs="EPSG:4326"
    )

    # True geodesic trip length:
    def geodesic_length(line):
        # line is a shapely LineString or MultiLineString in lon/lat
        if line.geom_type == "MultiLineString":
            segments = line.geoms
        else:
            segments = [line]
        length = 0.0
        for seg in segments:
            coords = list(seg.coords)
            lons, lats = zip(*coords)
            # returns (az12,az21,distances)
            length += sum(geod.line_lengths(lons, lats))
        return length

    gdf["trip_len_m"] = gdf.geometry.apply(geodesic_length)

    # 3) Save merged routes (projected) for mapping/hot counts
    gdf_proj = gdf.to_crs(32118)
    gdf_proj.to_file(RESULTS / f"merged_routes_neat_{hour_str}.gpkg", driver="GPKG")

    # 4) Load that hour’s UTCI segments (EPSG:32118)
    seg_fp = RESULTS / f"street_segments_utci_neat_{hh:02d}.gpkg"
    if not seg_fp.exists():
        print(f"   · missing UTCI file, skip"); continue
    streets = gpd.read_file(seg_fp).set_index("seg_id")[["geometry","utci"]].to_crs(32118)
    streets["seg_len"] = streets.geometry.length

    # 5) Hot-segment usage (modified overlap logic)
    buf = gdf_proj.geometry.buffer(BUF, cap_style=2)
    cand = gpd.sjoin(
        gpd.GeoDataFrame({"geometry": buf}, crs=32118),
        streets[["geometry"]],
        how="inner", predicate="intersects"
    ).rename(columns={"index_right": "seg_id"}).reset_index(drop=True)

    def qualifies(r):
        # length of the actual overlap
        inter_len = streets.geometry[r.seg_id].intersection(r.geometry).length
        # true if either ≥10 m OR ≥20% of the segment
        return (inter_len >= 10) or (inter_len / streets.seg_len[r.seg_id] >= MIN_RATIO)

    valid = cand[cand.apply(qualifies, axis=1)][["seg_id"]]
    usage_hr = valid.seg_id.astype(int).value_counts().rename("count")

    hot_segs    = streets[streets.utci >= HOT].index
    usage_hr_hot = usage_hr[usage_hr.index.isin(hot_segs)].astype(int)
    usage_global = usage_global.add(usage_hr_hot, fill_value=0)

    seg_use = streets.join(usage_hr_hot, how="left").fillna({"count": 0})
    seg_use["count"] = seg_use["count"].astype(int)
    top100 = seg_use[seg_use.utci >= HOT].nlargest(100, "count")
    top100.to_file(
        RESULTS / f"top100_high_utci_segments_neat_{hour_str}.gpkg",
        driver="GPKG"
    )
    print(f"   → wrote top100_high_utci_segments_neat_{hour_str}.gpkg")


    # 6) Compute length‐weighted mean UTCI per trip
    # Join back on buffered projected geometries
    joined = gpd.sjoin(
        gpd.GeoDataFrame({
            "ride_id":    gdf.ride_id,
            "trip_len_m": gdf.trip_len_m,
            "geometry":   gdf_proj.geometry
        }, crs=32118),
        streets[["geometry","utci","seg_len"]],
        how="inner", predicate="intersects"
    )
    if not joined.empty:
        def trip_stats(grp):
            return pd.Series({
                "trip_len_m": grp["trip_len_m"].iloc[0],
                "mean_utci":  (grp.seg_len * grp.utci).sum() / grp.seg_len.sum()
            })
        agg = joined.groupby("ride_id", sort=False).apply(trip_stats).reset_index()
        agg["hour"] = hh
        global_mean_recs.append(agg)
        agg.to_csv(OUT_DIR / f"mean_utci_{hour_str}.csv", index=False)
        print(f"   → saved mean_utci_{hour_str}.csv ({len(agg)} trips)")
    else:
        pd.DataFrame(columns=["ride_id","trip_len_m","mean_utci","hour"]).to_csv(
            OUT_DIR / f"mean_utci_{hour_str}.csv", index=False
        )
        print("   → mean-UTCI: 0 trips")

    print(f" processed {len(trips_hr):,} trips\n")

# 7) Final global summaries
usage_global.fillna(0, inplace=True)

df_all = pd.concat(global_mean_recs, ignore_index=True)
hourly_avg = (
    df_all.groupby("hour")
          .apply(lambda g: (g.mean_utci * g.trip_len_m).sum() / g.trip_len_m.sum())
          .rename("weighted_avg_utci")
          .reset_index()
)
hourly_avg.to_csv(OUT_DIR/"hourly_weighted_avg_utci.csv", index=False)

global_weighted = (df_all.mean_utci * df_all.trip_len_m).sum() / df_all.trip_len_m.sum()
global_median   = df_all.mean_utci.median()
pd.DataFrame({
    "metric": ["global_weighted_avg_utci","global_median_utci"],
    "value":  [global_weighted, global_median]
}).to_csv(OUT_DIR/"global_mean_utci_summary.csv", index=False)

print("Done! All outputs in:", OUT_DIR)


── Hour 0000 ──
   → wrote top100_high_utci_segments_neat_0000.gpkg


C:\Users\Agustin\AppData\Local\Temp\ipykernel_52544\2601128180.py:122: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg = joined.groupby("ride_id", sort=False).apply(trip_stats).reset_index()


   → saved mean_utci_0000.csv (74360 trips)
 processed 76,906 trips

── Hour 0100 ──
   → wrote top100_high_utci_segments_neat_0100.gpkg


C:\Users\Agustin\AppData\Local\Temp\ipykernel_52544\2601128180.py:122: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg = joined.groupby("ride_id", sort=False).apply(trip_stats).reset_index()


   → saved mean_utci_0100.csv (44575 trips)
 processed 46,271 trips

── Hour 0200 ──
   → wrote top100_high_utci_segments_neat_0200.gpkg


...

   → saved mean_utci_1400.csv (281052 trips)
 processed 289,398 trips

── Hour 1500 ──
   → wrote top100_high_utci_segments_neat_1500.gpkg


C:\Users\Agustin\AppData\Local\Temp\ipykernel_52544\2601128180.py:122: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg = joined.groupby("ride_id", sort=False).apply(trip_stats).reset_index()


   → saved mean_utci_1500.csv (300224 trips)
 processed 309,352 trips

── Hour 1600 ──
   → wrote top100_high_utci_segments_neat_1600.gpkg


C:\Users\Agustin\AppData\Local\Temp\ipykernel_52544\2601128180.py:122: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg = joined.groupby("ride_id", sort=False).apply(trip_stats).reset_index()


   → saved mean_utci_1600.csv (327411 trips)
 processed 337,020 trips

── Hour 1700 ──


## 9. Export Full Street Network with Global Hotspot Counts

This section combines the accumulated high-UTCI route usage counts with the base NEAT street network. The resulting file, `street_network_with_counts.gpkg`, is used to generate global hotspot layers such as the top 100, top 500, or top 1000 highest-use hot street segments.

In [ ]:
# ─── 9 · EXPORT FULL STREET NETWORK WITH GLOBAL HOTSPOT COUNTS ─────────

TMP_DIR = RESULTS_DIR / "temp_segments"
TMP_DIR.mkdir(parents=True, exist_ok=True)

usage_global = usage_global.fillna(0).astype(int)
usage_global.name = "count"

streets_with_counts = streets_base.set_index("seg_id").copy()
streets_with_counts["count"] = usage_global.reindex(streets_with_counts.index).fillna(0).astype(int)

network_fp = TMP_DIR / "street_network_with_counts.gpkg"
streets_with_counts.reset_index().to_file(network_fp, driver="GPKG")

print("Street network with global hotspot counts written to:", network_fp)

## 10. Export Global Top Hotspot Segments

This section aggregates hot-segment usage across all hours and exports the most frequently used high-UTCI street segments. Manual exclusions can be applied here if specific segments were identified as invalid artifacts during verification.

In [13]:
import geopandas as gpd
from pathlib import Path

# ─── CONFIG ───────────────────────────────────────────────────────────
TMP_DIR     = Path(r"C:\Users\Agustin\Documents\2025summer\sanity_check\results\temp_segments")
FULL_NET    = TMP_DIR / "street_network_with_counts.gpkg"
OUTPUT_FP   = TMP_DIR / "top1000_high_utci_segments_neat_GLOBAL.gpkg"

# Segment IDs to exclude
exclude_ids = {58251, 58267, 27, 53086, 56194, 58345, 10148, 53324, 9712, 62, 70, 58260, 12, 52736, 29566, 28997, 58347, 51655, 58339, 58341, 58237, 58238, 22084, 41173}

# ─── LOAD FULL NETWORK WITH COUNTS ────────────────────────────────────
net = gpd.read_file(FULL_NET).set_index("seg_id")
print(f"Loaded full network: {len(net)} segments")

# ─── EXCLUDE SPECIFIED SEGMENTS ─────────────────────────────────────
net_filtered = net.drop(labels=exclude_ids, errors="ignore")
print(f"After exclusion: {len(net_filtered)} segments")

# ─── SELECT TOP 1000 BY COUNT ─────────────────────────────────────────
top1000 = net_filtered.nlargest(1000, "count")
print(f"Selected top 1000 segments: {len(top1000)}")

# ─── WRITE OUT NEW TOP-1000 LAYER ─────────────────────────────────────
top1000.reset_index().to_file(OUTPUT_FP, driver="GPKG")
print("Wrote top-1000 cleaned segments to:", OUTPUT_FP)


Loaded full network: 58472 segments
After exclusion: 58457 segments
Selected top 1000 segments: 1000
Wrote top-1000 cleaned segments to: C:\Users\Agustin\Documents\2025summer\sanity_check\results\temp_segments\top1000_high_utci_segments_neat_GLOBAL.gpkg


This final export selects the highest-use hot street segments from the globally accumulated hotspot counts. The example below exports the top 1,000 verified hotspot segments after removing manually identified invalid or artifact segments. The same filtering and export process was repeated for other hotspot thresholds, including the top 100 and top 500 segments, to support salternative map outputs.